# 77 — Agent Loop Compiler Experiment (DSPy)

## Goal
Build an **agentic workflow** in DSPy — a module that
can decide to use tools in a loop — and **optimize**
its performance with evaluations.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **DSPy ReAct** | Built-in agent module with tool use |
| **Tool definition** | Python functions as agent tools |
| **Agent loop** | Observe → Think → Act → Observe cycle |
| **Optimization** | Compile agent behavior with examples |
| **Evaluation** | Measure agent task completion |

## Stack
- `dspy` 3.1+ (ReAct module, tools, optimizers)
- Ollama `qwen3.5:9b`

In [ ]:
import os, warnings, math
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

import dspy

lm = dspy.LM(
    "ollama_chat/qwen3.5:9b",
    api_base="http://localhost:11434",
    api_key="fake",
    temperature=0,
)
dspy.configure(lm=lm)
print(f"DSPy {dspy.__version__} configured")

## Step 1 — Define Agent Tools

We create tools for a math/data agent that can
perform calculations, look up data, and convert units.

In [ ]:
# Simulated data lookup
COMPANY_DATA = {
    "revenue_2024": 5_200_000,
    "revenue_2023": 4_100_000,
    "employees": 85,
    "customers": 1200,
    "churn_rate": 0.08,
    "avg_deal_size": 4333,
    "monthly_burn": 380_000,
    "cash_on_hand": 2_800_000,
}

def lookup_metric(metric_name: str) -> str:
    """Look up a company metric by name. Available metrics: revenue_2024, revenue_2023,
    employees, customers, churn_rate, avg_deal_size, monthly_burn, cash_on_hand."""
    key = metric_name.strip().lower().replace(" ", "_")
    if key in COMPANY_DATA:
        return f"{key} = {COMPANY_DATA[key]}"
    return f"Metric '{metric_name}' not found. Available: {list(COMPANY_DATA.keys())}"

def calculate(expression: str) -> str:
    """Evaluate a math expression safely. Examples: '5200000/4100000 - 1', '2800000/380000'."""
    allowed = set("0123456789.+-*/() ")
    if not all(c in allowed for c in expression):
        return f"Invalid expression: only numbers and +-*/() allowed"
    try:
        result = eval(expression)  # safe: only numbers/operators allowed
        return f"{expression} = {result:.4f}"
    except Exception as e:
        return f"Error: {e}"

def format_currency(amount: str) -> str:
    """Format a number as USD currency. Example input: '5200000'."""
    try:
        val = float(amount)
        if val >= 1_000_000:
            return f"${val/1_000_000:.1f}M"
        elif val >= 1_000:
            return f"${val/1_000:.1f}K"
        else:
            return f"${val:.2f}"
    except ValueError:
        return f"Cannot format '{amount}' as currency"

tools = [lookup_metric, calculate, format_currency]
print(f"Defined {len(tools)} tools: {[t.__name__ for t in tools]}")

# Test tools
print(lookup_metric("revenue_2024"))
print(calculate("5200000 / 4100000 - 1"))
print(format_currency("5200000"))

## Step 2 — Build DSPy ReAct Agent

`dspy.ReAct` implements the Reasoning+Acting loop:
the LLM decides which tool to call at each step.

In [ ]:
# Create DSPy tools from functions
dspy_tools = [dspy.Tool(fn) for fn in tools]

# Build ReAct agent
agent = dspy.ReAct(
    "question -> answer",
    tools=dspy_tools,
    max_iters=5,
)

print("ReAct agent created (max 5 iterations)")
print(f"Tools: {[t.name for t in dspy_tools]}")

In [ ]:
# Test: simple lookup
result = agent(question="What is the company's revenue for 2024?")
print(f"Answer: {result.answer}")

In [ ]:
# Test: multi-step reasoning
result = agent(question="What is the year-over-year revenue growth rate from 2023 to 2024?")
print(f"Answer: {result.answer}")

In [ ]:
# Test: complex question requiring multiple tools
result = agent(question="How many months of runway does the company have based on cash and burn rate?")
print(f"Answer: {result.answer}")

## Step 3 — Create Evaluation Dataset

In [ ]:
# Agent tasks with expected answers
agent_tasks = [
    {"question": "What is the company's 2024 revenue?",
     "answer": "5200000", "key_terms": ["5200000", "5.2"]},
    {"question": "How many employees does the company have?",
     "answer": "85", "key_terms": ["85"]},
    {"question": "What is the revenue growth rate from 2023 to 2024?",
     "answer": "26.83%", "key_terms": ["26", "27", "0.26", "0.27", "growth"]},
    {"question": "What is the revenue per employee?",
     "answer": "61176", "key_terms": ["61", "per employee"]},
    {"question": "How many months of cash runway remain?",
     "answer": "7.37 months", "key_terms": ["7", "month", "runway"]},
    {"question": "What is the number of customers?",
     "answer": "1200", "key_terms": ["1200"]},
]

examples = [
    dspy.Example(question=t["question"], answer=t["answer"]).with_inputs("question")
    for t in agent_tasks
]

trainset = examples[:4]
testset = examples[4:]

print(f"Train: {len(trainset)}, Test: {len(testset)}")

In [ ]:
# Metric: check if key terms appear in the answer
def agent_metric(example, prediction, trace=None):
    pred_text = str(prediction.answer).lower()
    # Find matching task
    for task in agent_tasks:
        if task["question"] == example.question:
            key_terms = task["key_terms"]
            hits = sum(1 for term in key_terms if term.lower() in pred_text)
            return hits / len(key_terms)
    return 0.0

# Evaluate baseline agent
def evaluate_agent(ag, dataset, name=""):
    scores = []
    print(f"\n{name}")
    print("=" * 60)
    for ex in dataset:
        pred = ag(question=ex.question)
        score = agent_metric(ex, pred)
        scores.append(score)
        icon = "✅" if score >= 0.5 else "❌"
        print(f"  {icon} [{score:.2f}] Q: {ex.question[:50]}")
        print(f"       A: {pred.answer[:100]}")
    avg = sum(scores) / len(scores) if scores else 0
    print(f"  Avg score: {avg:.2f}")
    return avg

baseline_score = evaluate_agent(agent, testset, "Baseline Agent")

## Step 4 — Optimize the Agent

In [ ]:
# Optimize with LabeledFewShot
optimizer = dspy.LabeledFewShot(k=3)

optimized_agent = optimizer.compile(
    student=dspy.ReAct("question -> answer", tools=dspy_tools, max_iters=5),
    trainset=trainset,
)

print("Agent optimization complete!")

In [ ]:
# Evaluate optimized agent
optimized_score = evaluate_agent(optimized_agent, testset, "Optimized Agent")

print(f"\n{'='*60}")
print(f"COMPARISON")
print(f"{'='*60}")
print(f"  Baseline:  {baseline_score:.2f}")
print(f"  Optimized: {optimized_score:.2f}")

In [ ]:
# Inspect what the agent does internally
print("\nInspecting agent trace:")
print("=" * 60)
result = optimized_agent(
    question="What is the revenue growth rate from 2023 to 2024?"
)
print(f"\nFinal answer: {result.answer}")
dspy.inspect_history(n=1)

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **dspy.ReAct** | Agent with Thought → Action → Observation loop |
| **dspy.Tool** | Wraps a Python function as an agent tool |
| **Agent loop** | LLM decides which tool to call at each step |
| **max_iters** | Controls maximum reasoning steps |
| **Optimization** | Few-shot demos teach the agent better patterns |

## ReAct Agent Loop
```
Question
  │
  ├─ Thought: "I need to look up revenue_2024"
  ├─ Action:  lookup_metric("revenue_2024")
  ├─ Observation: "revenue_2024 = 5200000"
  │
  ├─ Thought: "Now I need revenue_2023 to compute growth"
  ├─ Action:  lookup_metric("revenue_2023")
  ├─ Observation: "revenue_2023 = 4100000"
  │
  ├─ Thought: "Calculate growth rate"
  ├─ Action:  calculate("5200000/4100000 - 1")
  ├─ Observation: "= 0.2683"
  │
  └─ Answer: "Revenue grew ~26.8% YoY"
```

## Extending This Notebook
- Add more complex tools (web search, DB queries)
- Try `dspy.BootstrapFewShot` to generate agent traces
- Increase `max_iters` for more complex multi-step tasks
- Build a custom `dspy.Module` with explicit tool routing
- Add error recovery logic in tools